In [0]:
%pip install -U -qqqq mlflow databricks-openai databricks-agents
dbutils.library.restartPython()

In [0]:
import json
import mlflow
from databricks.sdk import WorkspaceClient
from databricks_openai import UCFunctionToolkit, DatabricksFunctionClient
# Import MLflow utilities for converting from chat completions to Responses API format
from mlflow.types.responses import output_to_responses_items_stream, create_function_call_output_item

# Enable automatic tracing for easier debugging
mlflow.openai.autolog()

# Get an OpenAI client configured to connect to Databricks model serving endpoints
openai_client = WorkspaceClient().serving_endpoints.get_open_ai_client()

# Load Databricks built-in tools (Python code interpreter)
client = DatabricksFunctionClient()
builtin_tools = UCFunctionToolkit(function_names=["system.ai.python_exec"], client=client).tools
for tool in builtin_tools:
  del tool["function"]["strict"]

def call_tool(tool_name, parameters):
  if tool_name == "system__ai__python_exec":
    return DatabricksFunctionClient().execute_function("system.ai.python_exec", parameters=parameters).value
  raise ValueError(f"Unknown tool: {tool_name}")

def call_llm(prompt):
  for chunk in openai_client.chat.completions.create(
    model="databricks-claude-3-7-sonnet",
    messages=[{"role": "user", "content": prompt}],
    tools=builtin_tools,
    stream=True
  ):
    yield chunk.to_dict()

def run_agent(prompt):
  """
  Send a user prompt to the LLM, and yield LLM + tool call responses
  The LLM is allowed to call the code interpreter tool if needed, to respond to the user
  """
  # Convert output into Responses API-compatible events
  for chunk in output_to_responses_items_stream(call_llm(prompt)):
    yield chunk.model_dump(exclude_none=True)
  # If the model executed a tool, call it and yield the tool call output in Responses API format
  if chunk.item.get('type') == 'function_call':
    tool_name = chunk.item["name"]
    tool_args = json.loads(chunk.item["arguments"])
    tool_result = call_tool(tool_name, tool_args)
    yield {"type": "response.output_item.done", "item": create_function_call_output_item(call_id=chunk.item["call_id"], output=tool_result)}

In [0]:
for output_chunk in run_agent("What is the square root of 429?"):
  print(output_chunk)

In [0]:
%sql
USE CATALOG learn_adb_fikrat;
USE bronze;
CREATE OR REPLACE FUNCTION python_udf_summarize(x DOUBLE, y DOUBLE)
RETURNS DOUBLE
LANGUAGE PYTHON
COMMENT 'Returns summary of two numeric parameters'
AS
$$
return x + y
$$;




In [0]:
# The code below is SQL, not Python. To fix the syntax and convert to Python Spark DataFrame API:

# Set the catalog and schema
spark.sql("USE CATALOG learn_adb_fikrat")
spark.sql("USE bronze")

# Create or replace a Python UDF in Databricks (registering via SQL is correct, but here's a Python version for Spark)
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

@udf(DoubleType())
def python_udf_summarize(x, y):
    """Returns summary of two numeric parameters"""
    return x + y

# Register the UDF for SQL usage
spark.udf.register("python_udf_summarize", python_udf_summarize)

In [0]:
%sql
CREATE OR REPLACE FUNCTION sql_udf_accident_stats()
RETURNS TABLE (killed INT, injured INT)
LANGUAGE SQL
COMMENT 'Returns number of killed and injured people in vehicle accidents'
RETURN 
SELECT killed, injured FROM test_crashes;

In [0]:
%sql

select * , cast(`NUMBER OF MOTORIST INJURED` as int), cast(`NUMBER OF MOTORIST KILLED` as int),
python_udf_change(cast(`NUMBER OF MOTORIST INJURED` as int), cast(`NUMBER OF MOTORIST KILLED` as int)) as total
 from  vehicle_crashes limit 10

In [0]:
%sql
CREATE table test_crashes
AS
select Collision_id, cast(`NUMBER OF MOTORIST INJURED` as int) as injured, cast(`NUMBER OF MOTORIST KILLED` as int) as killed, 0 AS Total  from  vehicle_crashes where cast(`NUMBER OF MOTORIST INJURED` as int) <>0
AND cast(`NUMBER OF MOTORIST KILLED` as int) <>0
limit 10

In [0]:
%sql
select * from test_crashes

In [0]:
% pip install  databricks_langchain

In [0]:
from databricks_langchain import ChatDatabricks
chat_model = ChatDatabricks(
    endpoint="databricks-dbrx-instruct",
    temperature=0.1,
    max_tokens=256,
    # See https://python.langchain.com/api_reference/community/chat_models/langchain_community.chat_models.databricks.ChatDatabricks.html for other supported parameters
)

In [0]:
from langchain.agents import create_agent
agent = create_agent(
    chat_model,
    tools=tools
)

In [0]:
from langchain.tools import tool
from langchain.agents import create_agent
# from langchain.agents import initialize_agent
from databricks_langchain import ChatDatabricks

# 1. Define the function to modify a UC table
@tool
def update_uc_table():
    """
    Update rows in a test_crashes table.
    """
    query = f"UPDATE learn_adb_fikrat.bronze.test_crashes SET total=killed+injured"
    spark.sql(query)
    return f"Table updated"

# 2. Register the function as a tool
# update_tool = Tool.from_function(
#     func=update_uc_table,
#     name="update_uc_table",
#     description="Update rows in a Unity Catalog table. Args: table_name, set_expr, where_expr."
# )
tools=[update_uc_table]
# 3. Add the tool to your agent
llm = ChatDatabricks(endpoint="databricks-claude-sonnet-4", temperature=0.1)
agent = create_agent(
    tools=tools,
    model=llm
)

# Example usage
agent.invoke({"messages": "Update the total column to sum of killed and injured  in table test_crashes"})

In [0]:
from langchain.tools import tool
from langchain.agents import create_agent


@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

@tool
def get_weather(location: str) -> str:
    """Get weather information for a location."""
    return f"Weather in {location}: Sunny, 72°F"

agent = create_agent(llm, tools=[search, get_weather])

In [0]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

system_msg = SystemMessage(content=[
            {
                "type": "text",
                "text": "You are an AI assistant tasked weather forecasts.",
            }])
llm = ChatDatabricks(endpoint="databricks-claude-sonnet-4", temperature=0.1,system_message=system_msg)

human_msg = HumanMessage("What is the weather in 'London'?")
agent.invoke({"messages": [human_msg]})

In [0]:
from langchain.agents import create_agent
from langchain.messages import SystemMessage, HumanMessage

literary_agent = create_agent(
    model=llm,
    system_prompt=SystemMessage(
        content=[
            {
                "type": "text",
                "text": "You are an AI assistant tasked with analyzing literary works.",
            },
            {
                "type": "text",
                "text": "<the entire contents of 'Pride and Prejudice'>",
                "cache_control": {"type": "ephemeral"}
            }
        ]
    )
)

result = literary_agent.invoke(
    {"messages": [HumanMessage("Analyze the major themes in 'Pride and Prejudice'.")]}
)